# 第10回: Sandbox

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session10/session10_sbx.ipynb)

---

# 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session10

%pip install -q -e ".[server]"
!curl -fsSL https://chatgpt.com/codex/install.sh | CODEX_NON_INTERACTIVE=1 sh

---

## 1. サンドボックス実行

### サンドボックスとは

サンドボックスは、実行するコードやコマンドに与える権限・資源・到達範囲をあらかじめ制限し、問題が起きても影響を内側へ閉じ込める仕組み。

エージェントにコマンド実行やコード実行の Tool を渡すと、実行される内容を事前に確定できなくなる。何が動くかはモデルの出力次第で、そこには次のような経路で危険な操作が混ざる。

- モデルが誤ったコマンドを生成する（対象の取り違え、範囲の広すぎるパス指定）
- 読み込んだ文書や Web ページに含まれるプロンプトインジェクションに従う
- 取得したコードや依存パッケージが、意図しない動作を持ち込む

いずれも悪意ある利用者がいなくても起こる。エージェントが扱う入力の一部は外部から来るため、コマンドが生成されるまでの経路全体を信頼できる前提には置けない。


### 隔離の段階

隔離は 0 か 1 ではなく、強さに段階がある。上ほど軽く、下ほど強い。

| 隔離の方法 | 境界になるもの | 防げないこと |
|---|---|---|
| 作業ディレクトリを変える（`subprocess.run(..., cwd=...)`） | なし（ホストと同じ権限のまま） | 絶対パスでの読み書き、ネットワーク、資源の使い切り |
| 別プロセス＋資源上限（`rlimit`） | プロセスの CPU とメモリ | ファイルアクセス、ネットワーク |
| OS のサンドボックス機構（seccomp、Landlock、Seatbelt） | syscall とファイルパス | カーネルの脆弱性、プロファイルの設定漏れ |
| コンテナ | 名前空間、cgroup、capability | ホストと共有するカーネルの脆弱性 |
| microVM、専用 VM | 仮想化されたカーネル | ハイパーバイザの脆弱性 |
| 別アカウント・別ネットワークの実行環境 | クラウドの信頼境界 | 運用や設定の誤り |

強い隔離ほど起動が重く、内側で使えるツールも減る。



---

## 2. サンドボックス化したシェルの実装

[![](https://mermaid.ink/img/pako:eNqVlW1vmzAQx78Kct90EokCARLQNKlaJm1SuxdJXq1UyIVLQDE2cpwlLMp3n3EKMQ-tVEuBnP07--5_HJxRzBJAAdoQdoxTzIXxuAx5SA059kLa988hWq0flusQvXypVwjDSZRDznh5HyLNClHDwElwHIsb1p7QyL-YZAkWcEM7Mxp75JkO6qZGRUCTKvAfvxd12E1Sh9ctx0VqLOEhFvVsNZKMQywyRo31Qp9XMkQcZOyDYlTjFTaMQ4S3QIUMa5UCIWvGyFOWJASOmMNYR7RQq5HLGhDppu6dNSF32cs1de-s4Y0A_uGhGtHxlQrdctJ0arKudTrKa6Qii2JMiPGclBTnWRwVnOV5IcYd4EXfpcmuc7Ru9jOq9fh5yDH9RdcpPDJW9NIakuudwhmj0bdWkd4rngI7GelsV40K16PpVs74-gHR3Uyx10IPa6Id18tCm1ZYU-CmIW-ytzpc0XoP95tcIa2GUcb1nHZbD3a_ArtNPdz9bwXIepw-p6DoLaF-UiWBnrb8FFi2ycvqKhHOdjBK8F6-8jguA8M13La_rvogbhqbjJDAuHMcZ8j1WolPurYexU_6XkuiVu9sW2appdzUNKO7laJtc2rkmO-Aj6SOAWUUdCGrHzLRlmcJCgQ_gIly4DmuTHSuOPlKSiGHEAXybyJ3ClFIL9KnwPQPY3ntxtlhm9bGoahKvciwfLfcCBkA8O_sQAUKbN9TW6DgjE4oGE3d8dR3HXc-93zL8ya2iUo5bbnjiTefzHzfnnuOa1nuxUT_1LHW2PZcf-ZLj4nrSGBmIkgywfjT9VunPnmX_1K2Qko?type=png)](https://mermaid.live/edit#pako:eNqVlW1vmzAQx78Kct90EokCAQpomlQtkzap3Yskr1Yq5MIloBgbOWZJVuW7zziFmIdWqqVAzv6dffc_Dl5RwlJAIdoQdkgyzIXxsIx4RA059kLat08RWq3vl-sIPX9pVgjDaVxAwfjpNkKaFaGWgaPgOBFXrDuhkX8xyVMs4Ir2ZjT2wHMd1E2NioGmdeA_fi-asNukqpctx2VmLOE-Ec1sPdKcQyJyRo31Qp9XMsQcZOyjYtTjBTaMQ4y3QIUMa5UBIWvGyGOepgQOmMNUR7RQ61HIGhDppu69NSF32cs1de-t4Y0A_uGhGtHzlQpdc9J0arNudDrIa6wiixNMiPGUnigu8iQuOSuKUkx7wLO-S5td72jdHGbU6PGzKjD9RdcZPDBWDtIak-udwhmTybdOkd4rngJ7GelsX40a16PpV874-gHR30yxl0KPa6IdN8hCm1ZYW-C2Ia-ydzpc0XoPD5tcIZ2GUcblnG5bj3a_AvtNPd79bwXIB5w-p6D4LaFhUicCA235MbRsk5_qq0Q428EkxXv5yuP4FBqu4Xb9ddVHcdPY5ISExo3jOGOul0p80rXzKH7S91IStXpj2zJLLeW2pjndrRRtm3OjwHwHfCJ1DCmjoAtZ_5CJtjxPUSh4BSYqgBe4NtFrzclXUgYFRCiUf1O5U4QiepY-JaZ_GCsaN86qbYbCDSZ7aVVlXetFjuXL5YrICIB_ZxUVKLQDS-2Bwld0ROFk7k7ngeu4vu8FlufNbBOd5LTlTmeeP7sLAtv3HNey3LOJ_qlzrantucFd4AdzZ-57rmsiSHPB-OPlW6c-eef_VPdCWQ)

エージェントのコマンド実行を隔離コンテナへ閉じ込める。

`ShellToolMiddleware` は、永続シェルセッションを `shell` Tool として ReAct ループへ登録する Middleware。`before_agent` でセッションを起動し、`after_agent` で片付ける。セッションが永続するため、`cd` や環境変数の設定が後続のコマンドへ引き継がれる。

コマンドをどこで動かすかは **ExecutionPolicy** で差し替える。Tool の名前と引数（`shell`／コマンド文字列）は変わらないので、エージェントの実装もプロンプトの組み立ても共通のまま、隔離の強さだけを入れ替えられる。

| ExecutionPolicy | 実行場所 | 隔離されるもの | 想定する用途 |
|---|---|---|---|
| `HostExecutionPolicy`（既定） | ホストのプロセス | CPU 時間とメモリのみ（`rlimit`） | すでにコンテナや VM で隔離済みの信頼環境 |
| `CodexSandboxExecutionPolicy` | Codex CLI のサンドボックス | syscall とファイルアクセス | Codex CLI があり、ホスト実行のまま制限を強めたい場合 |
| `DockerExecutionPolicy` | 専用の Docker コンテナ | ファイル、ネットワーク、権限、資源 | 未信頼の入力を扱う場合 |

今回は下表の設定のもとで `DockerExecutionPolicy`を使用する。

| 制約 | 設定 | `DockerExecutionPolicy` の指定 |
|---|---|---|
| ホストのファイル | bind mount しない | 作業ディレクトリを渡さない（一時ディレクトリはマウントされない） |
| ルートファイルシステム | 読み取り専用 | `read_only_rootfs=True` |
| 一時書き込み | サイズ制限付き `/tmp` のみ | `extra_run_args` の `--tmpfs` |
| ネットワーク | 無効 | `network_enabled=False` |
| 実行ユーザー | 非 root | `user='65534:65534'` |
| 権限 | capability 全削除、権限昇格禁止 | `extra_run_args` の `--cap-drop=ALL`、`--security-opt no-new-privileges=true` |
| 資源 | メモリ、CPU、プロセス数を制限 | `memory_bytes`、`cpus`、`extra_run_args` の `--pids-limit` |
| 実行時間 | コマンドごとに上限を設ける | `command_timeout` |
| 結果 | 行数とバイト数で切り詰める | `max_output_lines`、`max_output_bytes` |
| ライフサイクル | 実行終了時に破棄 | `remove_container_on_exit=True`（既定） |

実装手順:

1. ExecutionPolicy を設定した `ShellToolMiddleware` を追加する
2. `run_command` と `python_repl` を 除外する
3. `HumanInTheLoopMiddleware` の承認対象に `shell` を加える

In [ ]:
# @title 利用できるExecutionPolicyの確認
import shutil
import subprocess

SANDBOX_IMAGE = 'python:3.12-alpine'

# 後片付けのときに、この教材が起動したコンテナだけを選べるようにする
SANDBOX_LABEL = 'ai-agent-seminar=session09'

docker_cli = shutil.which('docker')
codex_cli = shutil.which('codex')

# host: 追加の依存が無いので常に使えるが、隔離もされない
print('host : 利用できます（隔離はされません）')

# codex: Codex CLI が必要。コンテナ内では Landlock が使えず起動に失敗することもある
if codex_cli is None:
    print('codex: Codex CLI が見つかりません')
else:
    print('codex: 利用できます:', codex_cli)

# docker: CLI とイメージの両方が必要
if docker_cli is None:
    print('docker: Docker CLI が見つかりません')
else:
    image_probe = subprocess.run(
        [docker_cli, 'image', 'inspect', SANDBOX_IMAGE],
        capture_output=True,
        text=True,
    )
    if image_probe.returncode == 0:
        print('docker: 利用できます:', SANDBOX_IMAGE)
    else:
        print(f'docker: イメージがありません。先に実行してください: docker pull {SANDBOX_IMAGE}')

In [ ]:
# @title 3つのExecutionPolicyの定義
from langchain.agents.middleware import (
    CodexSandboxExecutionPolicy,
    DockerExecutionPolicy,
    HostExecutionPolicy,
)

# どの Policy にも共通する上限。実行時間と、モデルへ返す観測の大きさを抑える
COMMON_LIMITS = {
    # コマンド1件あたりの実行時間の上限。超えるとセッションを再起動する
    'command_timeout': 15.0,
    'startup_timeout': 60.0,
    # モデルへ返す観測を切り詰める
    'max_output_lines': 50,
    'max_output_bytes': 4_000,
}

# --- Docker: 専用コンテナへ隔離する -------------------------------------------
docker_policy = DockerExecutionPolicy(
    image=SANDBOX_IMAGE,
    # ネットワークを無効にする（--network none）
    network_enabled=False,
    # ルートファイルシステムを読み取り専用にする（--read-only）
    read_only_rootfs=True,
    # 非 root ユーザー（nobody）で実行する
    user='65534:65534',
    # 資源の上限
    memory_bytes=256 * 1024 * 1024,
    cpus='0.5',
    # Policy が引数を持たない設定は docker run のオプションで補う
    extra_run_args=(
        f'--label={SANDBOX_LABEL}',
        '--tmpfs', '/tmp:rw,noexec,nosuid,nodev,size=64m,mode=1777',
        '--cap-drop=ALL',
        '--security-opt', 'no-new-privileges=true',
        '--pids-limit=64',
    ),
    **COMMON_LIMITS,
)

# --- Codex: Codex CLI のサンドボックスへ委ねる ---------------------------------
codex_policy = CodexSandboxExecutionPolicy(
    # 'auto' は OS を見て Seatbelt（macOS）と Landlock/seccomp（Linux）を選ぶ
    platform='auto',
    # codex の -c オプションへ渡す設定。ここでは外向き通信を止める
    # 新しい権限プロファイル（[permissions]）を使う設定では、こちらは無視される
    config_overrides={'sandbox_workspace_write.network_access': False},
    # Policy 自体は資源制限を持たないため、cgroup などホスト側の仕組みと併用する
    **COMMON_LIMITS,
)

# --- Host: ホストプロセスでそのまま動かす -------------------------------------
host_policy = HostExecutionPolicy(
    # ファイルもネットワークも隔離されない。守れるのは CPU 時間とメモリだけ
    cpu_time_seconds=10,
    memory_bytes=1024 * 1024 * 1024,
    **COMMON_LIMITS,
)

In [ ]:
# @title ShellToolMiddlewareの設定
from dataclasses import dataclass
from typing import Any

from langchain.agents.middleware import ShellToolMiddleware


@dataclass
class ShellSessionSpec:
    """ExecutionPolicy ごとに変わるシェルセッションの設定。

    シェルの実体（bash / sh）と作業ディレクトリの初期化は Policy によって変わる。
    note はサンドボックスの制約をモデルへ伝えるための説明文。
    """

    policy: Any
    shell_command: str
    startup_commands: tuple[str, ...]
    note: str


SHELL_SESSION_SPECS = {
    'docker': ShellSessionSpec(
        policy=docker_policy,
        # alpine イメージに bash は無い。既定の /bin/bash では起動に失敗する
        shell_command='/bin/sh',
        # 書き込める /tmp を作業ディレクトリにする。永続セッションなので cd は引き継がれる
        startup_commands=('cd /tmp',),
        note="""shell Tool は隔離コンテナ内の /bin/sh セッションで、次の制約がある。
- bash は無い。POSIX sh の構文で書く
- 書き込めるのは /tmp だけ。ルートファイルシステムは読み取り専用
- 外部ネットワークへは接続できない""",
    ),
    'codex': ShellSessionSpec(
        policy=codex_policy,
        shell_command='/bin/bash',
        # 作業ディレクトリは一時ディレクトリが割り当てられるので cd は不要
        startup_commands=(),
        note="""shell Tool は Codex CLI のサンドボックス内の /bin/bash セッション。
- ホスト上で動くが、syscall とファイルアクセスが制限されている
- 書き込めるのは作業ディレクトリと /tmp だけ
- 外部ネットワークへは接続できない""",
    ),
    'host': ShellSessionSpec(
        policy=host_policy,
        shell_command='/bin/bash',
        startup_commands=(),
        note="""shell Tool はホスト上の /bin/bash セッションで、隔離されていない。
- 作業ディレクトリは一時ディレクトリ。その外のファイルは変更しない
- 外部ネットワークへ接続できる""",
    ),
}

execution_policy_name = 'docker' #@param ['docker', 'codex', 'host']
spec = SHELL_SESSION_SPECS[execution_policy_name]

# 前提のCLIが無いまま進むと、グラフ実行の途中でセッション起動に失敗する
required_cli = {'docker': docker_cli, 'codex': codex_cli}.get(execution_policy_name)
if execution_policy_name in ('docker', 'codex') and required_cli is None:
    raise RuntimeError(
        f'{execution_policy_name} を使うにはCLIが必要です。'
        '先のセルの確認結果を見て、利用できるものを選んでください。'
    )

shell_middleware = ShellToolMiddleware(
    execution_policy=spec.policy,
    shell_command=spec.shell_command,
    startup_commands=spec.startup_commands,
    # workspace_root を渡さないため、ホストの作業ディレクトリは bind mount されない
    # env を渡さないため、APIキーなどの環境変数もセッションへ渡らない
)

print('ExecutionPolicy:', type(spec.policy).__name__)
print('追加されるTool:', [tool.name for tool in shell_middleware.tools])

In [ ]:
# @title ホスト実行Toolの除外と承認対象の更新
from langchain.agents.middleware import HumanInTheLoopMiddleware
from guarded_agent import DEFAULT_TOOLS

# ホストでそのまま実行される Tool は外す。残すと sandbox を迂回できてしまう
EXCLUDED_TOOLS = {'run_command', 'python_repl'}
TOOLS = [tool for tool in DEFAULT_TOOLS if tool.name not in EXCLUDED_TOOLS]

# 承認対象に shell, write_file, file_delete を追加する
hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        'shell': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'write_file': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'file_delete': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'read_file': False,
    },
    description_prefix='Toolの実行には承認が必要',
)

print('エージェントへ渡すTool:', [tool.name for tool in TOOLS] + ['shell'])

In [ ]:
# @title 長期記憶グラフの状態と読み込みノード
from dataclasses import dataclass
from typing_extensions import NotRequired

from langchain.agents.middleware import AgentState
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


class LongTermMemoryState(AgentState):
    memories: NotRequired[list[str]]
    memory_candidates: NotRequired[list[str]]
    approved_memories: NotRequired[list[str]]

In [ ]:
# @title 長期記憶グラフへサンドボックス付きエージェントを組み込む
from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langchain_core.runnables.graph_mermaid import CurveStyle
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from guarded_agent.state import Context
from guarded_agent.memory import load_memory, make_extract_memory, validate_memory, write_memory

model_id = 'gpt-5.4-nano' #@param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
model = ChatOpenAI(model=model_id)

long_term_store = InMemoryStore(
    index={
        'embed': OpenAIEmbeddings(model='text-embedding-3-small'),
        'dims': 1536,
        'fields': ['text'],
    }
)

@dynamic_prompt
def sandbox_system_prompt(request: ModelRequest) -> str:
    memories = request.state.get('memories', [])
    prompt = f"""あなたはAIエージェントです。
必要な場合はToolを利用してください。

コマンドやコードの実行には shell Tool だけを使うこと。
出力が改行で終わるコマンドを使うこと。
{spec.note}
"""
    if memories:
        memory_text = '\n'.join(f'- {memory}' for memory in memories)
        prompt += f"""
参考になる長期記憶:
{memory_text}
"""
    return prompt


sandbox_agent = create_agent(   
    model=model,
    tools=TOOLS,
    state_schema=LongTermMemoryState,
    middleware=[sandbox_system_prompt, shell_middleware, hitl_middleware],
)

sandbox_builder = StateGraph(
    LongTermMemoryState,
    context_schema=Context,
)
sandbox_builder.add_node('load_memory', load_memory)
sandbox_builder.add_node('agent', sandbox_agent)
sandbox_builder.add_node('extract_memory', make_extract_memory(model))
sandbox_builder.add_node('validate_memory', validate_memory)
sandbox_builder.add_node('write_memory', write_memory)

sandbox_builder.add_edge(START, 'load_memory')
sandbox_builder.add_edge('load_memory', 'agent')
sandbox_builder.add_edge('agent', 'extract_memory')
sandbox_builder.add_edge('extract_memory', 'validate_memory')
sandbox_builder.add_edge('validate_memory', 'write_memory')
sandbox_builder.add_edge('write_memory', END)

# 長期記憶の Store は前半と共有する
graph_with_sandbox = sandbox_builder.compile(
    store=long_term_store,
    checkpointer=InMemorySaver(),
)

display(
    Image(
        graph_with_sandbox.get_graph(xray=2).draw_mermaid_png(curve_style=CurveStyle.NATURAL)
    )
)

In [ ]:
# @title shellの実行を承認する
import json
from uuid import uuid4

sandbox_config = {
    'configurable': {'thread_id': f'sandbox-agent-{uuid4()}'}
}
context = Context(
    user_id='test',
    model='gpt-5.4-nano',
)
pending_shell = graph_with_sandbox.invoke(
    {
        'messages': [
            HumanMessage(
                content='1から100までの素数の個数を、shell Toolで計算してください。'
            )
        ]
    },
    config=sandbox_config,
    context=context,
)

review = pending_shell['__interrupt__'][0].value
print(json.dumps(review, ensure_ascii=False, indent=2))

In [ ]:
# @title 承認してサンドボックス内で実行する
from langgraph.types import Command

def approve_all(state, config, *, max_rounds=5):
    """承認を求められる限り approve で再開する。

    ReActループはツールを何度も呼ぶため、1回の依頼で承認要求が複数回発生しうる。
    実行が終わると `__interrupt__` が無くなる。
    """
    for _ in range(max_rounds):
        if '__interrupt__' not in state:
            return state
        request = state['__interrupt__'][0].value['action_requests'][0]
        print('承認したコマンド:', request['args'].get('command'))
        state = graph_with_sandbox.invoke(
            Command(resume={'decisions': [{'type': 'approve'}]}),
            config=config,
            context=context,
        )
    raise RuntimeError(f'承認が{max_rounds}回を超えました')


approved_shell = approve_all(pending_shell, sandbox_config)

# ツールの観測（サンドボックスの出力）と最終回答を確認する
for message in approved_shell['messages'][-2:]:
    message.pretty_print()

In [ ]:
# @title コマンドを修正して実行する
# レビュー担当者が実行内容を確定させる。承認した文字列がそのままシェルへ渡る。
# 同じコマンドでも、選んだ ExecutionPolicy によって結果が変わる
SANDBOX_PROBE_COMMAND = (
    'id; '
    'echo sandboxed > /tmp/result.txt && echo "tmp: $(cat /tmp/result.txt)"; '
    "echo blocked > /blocked.txt 2>/dev/null "
    "&& echo 'rootfs: 書き込めた' || echo 'rootfs: 書き込み不可'; "
    'wget -q -T 3 -O- https://example.com > /dev/null 2>&1 '
    "&& echo 'network: 到達' || echo 'network: 到達不可'"
)

edit_config = {
    'configurable': {'thread_id': f'sandbox-agent-{uuid4()}'}
}

pending_edit = graph_with_sandbox.invoke(
    {
        'messages': [
            HumanMessage(
                content=(
                    'shell Toolで、このサンドボックスの実行ユーザーと'
                    'アクセスできる範囲を確認してください。'
                )
            )
        ]
    },
    config=edit_config,
    context=context,
)

print('モデルの提案:', pending_edit['__interrupt__'][0].value['action_requests'][0]['args'])

edited_shell = graph_with_sandbox.invoke(
    Command(
        resume={
            'decisions': [{
                'type': 'edit',
                'edited_action': {
                    'name': 'shell',
                    'args': {'command': SANDBOX_PROBE_COMMAND},
                },
            }],
        }
    ),
    config=edit_config,
    context=context,
)

# 修正後もモデルが追加のコマンドを求めることがあるため、残りは承認で進める
edited_shell = approve_all(edited_shell, edit_config)

for message in edited_shell['messages'][-2:]:
    message.pretty_print()

In [ ]:
# @title サンドボックスコンテナの後片付け
# DockerExecutionPolicy を使ったときだけ意味がある。
# 中断したまま再開しなかった実行のコンテナは、参照が切れた時点で片付けられる
import gc

gc.collect()

if docker_cli is None:
    print('Docker CLI が無いため、後片付けするコンテナはありません')
else:
    leftover = subprocess.run(
        [docker_cli, 'ps', '--quiet', '--filter', f'label={SANDBOX_LABEL}'],
        capture_output=True,
        text=True,
    ).stdout.split()

    if leftover:
        subprocess.run(
            [docker_cli, 'rm', '--force', *leftover],
            capture_output=True,
            text=True,
        )

    print('強制削除したサンドボックスコンテナ:', len(leftover))

---
## 3. 保護されたエージェント: `guarded_agent`

In [ ]:
!echo "OPENAI_API_KEY=$OPENAI_API_KEY" >> .env
!echo "TAVILY_API_KEY=$TAVILY_API_KEY" >> .env

In [ ]:
if IS_COLAB:
    %cd /content/ai-agent-seminar/guarded_agent
    !langgraph dev --tunnel
else:
    !langgraph dev